# Etapa 1: Aprendizado Simbólico (Árvore de decisão)
**Pergunta prática:** A escola possui infraestrutura digital básica?
**Domínio:** Educação (Censo Escolar / INEP - Amazonas)

In [13]:



import pandas as pd
import os

# ---------------------------------------------------------------
# Pré-processamento: Leitura e Filtro dos Dados do INEP (Amazonas)
# ---------------------------------------------------------------

CAMINHO_ENTRADA = "Tabela_Escola_2025_V2.csv"
CAMINHO_SAIDA = "dados_etapa1.csv"

COL_ID = "CO_ENTIDADE"

COLS_FILTRO = [
    "SG_UF",
    "TP_SITUACAO_FUNCIONAMENTO"
]

COLS_PISTAS = [
    "IN_LABORATORIO_INFORMATICA",
    "IN_BIBLIOTECA",
    "TP_LOCALIZACAO",
    "TP_DEPENDENCIA",
    "IN_ENERGIA_GERADOR_FOSSIL",
]

COLS_ALVO = [
    "IN_INTERNET_ALUNOS",
    "IN_DESKTOP_ALUNO",
    "IN_COMP_PORTATIL_ALUNO",
    "IN_TABLET_ALUNO",
]

colunas = [COL_ID] + COLS_FILTRO + COLS_PISTAS + COLS_ALVO

# ---------------------------------------------------------------
# Processamento em partes (chunks)
# ---------------------------------------------------------------

TAMANHO_CHUNK = 50000

total_escolas = 0
primeiro_chunk = True

# Remove o arquivo anterior, caso exista
if os.path.exists(CAMINHO_SAIDA):
    os.remove(CAMINHO_SAIDA)

for df in pd.read_csv(
    CAMINHO_ENTRADA,
    sep=";",
    encoding="latin-1",
    usecols=colunas,
    chunksize=TAMANHO_CHUNK,
    on_bad_lines="skip"
):

    # Filtro para escolas do Amazonas em funcionamento
    df = df[
        (df["SG_UF"] == "AM") &
        (pd.to_numeric(
            df["TP_SITUACAO_FUNCIONAMENTO"],
            errors="coerce"
        ) == 1)
    ].copy()

    # Conversão das variáveis numéricas
    for coluna in COLS_PISTAS + COLS_ALVO:
        df[coluna] = pd.to_numeric(
            df[coluna],
            errors="coerce"
        )

    # Tratamento de valores ausentes
    df[COLS_ALVO] = df[COLS_ALVO].fillna(0)
    df = df.dropna(subset=COLS_PISTAS).copy()

    # Criação da variável-alvo binária
    tem_aparelho = (
        (df["IN_DESKTOP_ALUNO"] == 1)
        | (df["IN_COMP_PORTATIL_ALUNO"] == 1)
        | (df["IN_TABLET_ALUNO"] == 1)
    )

    tem_internet = df["IN_INTERNET_ALUNOS"] == 1

    df["infra_digital_basica"] = (
        tem_internet & tem_aparelho
    ).astype(int)

    # Seleção das colunas finais
    saida = df[
        [COL_ID] + COLS_PISTAS + ["infra_digital_basica"]
    ]

    # Salva cada parte no arquivo de saída
    saida.to_csv(
        CAMINHO_SAIDA,
        sep=",",
        index=False,
        encoding="utf-8",
        mode="w" if primeiro_chunk else "a",
        header=primeiro_chunk
    )

    primeiro_chunk = False
    total_escolas += len(saida)

# ---------------------------------------------------------------
# Resultado final
# ---------------------------------------------------------------

if primeiro_chunk:
    pd.DataFrame(
        columns=[COL_ID] + COLS_PISTAS + ["infra_digital_basica"]
    ).to_csv(CAMINHO_SAIDA, index=False, encoding="utf-8")

print("Pré-processamento concluído!")
print(f"Arquivo gerado: {CAMINHO_SAIDA}")
print(f"Total de escolas: {total_escolas}")
print(f"Total de colunas: {len(colunas) - len(COLS_FILTRO) + 1 - 0}")


Pré-processamento concluído!
Arquivo gerado: dados_etapa1.csv
Total de escolas: 5562
Total de colunas: 11


## 3. Cálculos Manuais de Entropia e Ganho de Informação

Nesta etapa, realizamos o cálculo manual da entropia e do ganho de informação para os três primeiros níveis da árvore de decisão, utilizando os microdados filtrados do Amazonas ($N = 5.562$ escolas)span_4span_4span_5span_5span_6span_6.

### A. Entropia do Nó Raiz (Nível 0)

- **Total de exemplos ($N$):** $5.562$span_7span_7
- **Classe 0 (Inadequado):** $4.555$ escolas ($\approx 81,90\%$)span_8span_8
- **Classe 1 (Adequado):** $1.007$ escolas ($\approx 18,10\%$)span_9span_9
- **Entropia Inicial ($E(S)$):** $0,68237$span_10span_10

### B. Ganho de Informação na Raiz (Atributos Candidatos)

- **`IN_LABORATORIO_INFORMATICA`**: **0,19153** *(Selecionado por apresentar o maior ganho)*span_11span_11span_12span_12
- **`IN_BIBLIOTECA`**: $0,09893$span_13span_13
- **`TP_LOCALIZACAO`**: $0,09528$span_14span_14
- **`TP_DEPENDENCIA`**: $0,04378$span_15span_15
- **`IN_ENERGIA_GERADOR_FOSSIL`**: $0,00878$span_16span_16

### C. Níveis Seguintes

- **Nível 1 (Filhos de `IN_LABORATORIO_INFORMATICA`):**
    - Para $Lab = 0.0$, o atributo selecionado foi **`TP_LOCALIZACAO`** ($IG = 0,01869$)span_17span_17span_18span_18.
    - Para $Lab = 1.0$, o atributo selecionado foi **`TP_DEPENDENCIA`** ($IG = 0,05249$)span_19span_19span_20span_20.

In [17]:

import csv
import math
from collections import Counter

# ---------------------------------------------------------------------------
# 1. Medidas de impureza
# ---------------------------------------------------------------------------

def entropia(exemplos, alvo):
    """H(S) = -sum_i p_i log2(p_i)"""
    n = len(exemplos)

    if n == 0:
        return 0.0

    contagem = Counter(e[alvo] for e in exemplos)

    return -sum(
        (c / n) * math.log2(c / n)
        for c in contagem.values()
    )


def ganho_informacao(exemplos, atributo, alvo):
    """GI(S, A) = H(S) - sum_v |S_v|/|S| * H(S_v)"""
    n = len(exemplos)
    grupos = {}

    for e in exemplos:
        grupos.setdefault(e[atributo], []).append(e)

    resto = sum(
        len(g) / n * entropia(g, alvo)
        for g in grupos.values()
    )

    return entropia(exemplos, alvo) - resto


def valor_plural(exemplos, alvo):
    """Classe mais frequente (empate -> menor valor, determinístico)."""
    contagem = Counter(e[alvo] for e in exemplos)

    maior = max(contagem.values())

    return min(
        c for c, k in contagem.items()
        if k == maior
    )


# ---------------------------------------------------------------------------
# 2. Estrutura da árvore
# ---------------------------------------------------------------------------

class No:
    """Nó interno (atributo + filhos) ou folha (classe)."""

    def __init__(
        self,
        atributo=None,
        filhos=None,
        classe=None,
        n=0,
        contagem=None
    ):
        self.atributo = atributo
        self.filhos = filhos or {}
        self.classe = classe
        self.n = n
        self.contagem = contagem or {}

    @property
    def eh_folha(self):
        return self.atributo is None


def learn_decision_tree(
    exemplos,
    atributos,
    exemplos_pai,
    alvo,
    dominio,
    profundidade=0,
    max_depth=None,
    historico=None
):
    """
    exemplos      : exemplos que chegaram a este nó
    atributos    : atributos ainda disponíveis
    exemplos_pai : exemplos do nó pai
    alvo         : variável-alvo
    dominio      : valores possíveis de cada atributo
    historico    : registra o ganho de cada atributo em cada nó
    """

    # Caso 1: sem exemplos -> classe plural do pai
    if not exemplos:
        return No(
            classe=valor_plural(exemplos_pai, alvo),
            n=0
        )

    contagem = dict(
        Counter(e[alvo] for e in exemplos)
    )

    # Caso 2: todos os exemplos têm a mesma classe -> folha pura
    if len(contagem) == 1:
        return No(
            classe=next(iter(contagem)),
            n=len(exemplos),
            contagem=contagem
        )

    # Caso 3: sem atributos ou profundidade máxima -> classe plural
    if (
        not atributos
        or (
            max_depth is not None
            and profundidade >= max_depth
        )
    ):
        return No(
            classe=valor_plural(exemplos, alvo),
            n=len(exemplos),
            contagem=contagem
        )

    # Caso 4: escolhe o atributo de maior ganho de informação
    ganhos = {
        a: ganho_informacao(exemplos, a, alvo)
        for a in atributos
    }

    # Empate -> primeiro atributo da lista
    melhor = max(atributos, key=lambda a: ganhos[a])

    if historico is not None:
        historico.append({
            "profundidade": profundidade + 1,
            "n": len(exemplos),
            "contagem": contagem,
            "H": entropia(exemplos, alvo),
            "escolhido": melhor,
            "ganhos": ganhos
        })

    no = No(
        atributo=melhor,
        classe=valor_plural(exemplos, alvo),
        n=len(exemplos),
        contagem=contagem
    )

    restantes = [
        a for a in atributos
        if a != melhor
    ]

    # Um ramo para cada valor possível do atributo
    for v in sorted(dominio[melhor]):
        sub = [
            e for e in exemplos
            if e[melhor] == v
        ]

        no.filhos[v] = learn_decision_tree(
            sub,
            restantes,
            exemplos,
            alvo,
            dominio,
            profundidade + 1,
            max_depth,
            historico
        )

    return no


def treinar(
    exemplos,
    atributos,
    alvo,
    max_depth=None,
    historico=None
):
    dominio = {
        a: {e[a] for e in exemplos}
        for a in atributos
    }

    return learn_decision_tree(
        exemplos,
        list(atributos),
        exemplos,
        alvo,
        dominio,
        0,
        max_depth,
        historico
    )


# ---------------------------------------------------------------------------
# 3. Predição, avaliação e impressão
# ---------------------------------------------------------------------------

def prever(no, exemplo):
    while not no.eh_folha:

        proximo = no.filhos.get(
            exemplo[no.atributo]
        )

        if proximo is None:
            # Valor nunca visto no treinamento
            return no.classe

        no = proximo

    return no.classe


def acuracia(no, exemplos, alvo):
    if not exemplos:
        return 0.0

    return sum(
        prever(no, e) == e[alvo]
        for e in exemplos
    ) / len(exemplos)


def imprimir(no, recuo=""):
    if no.eh_folha:
        print(
            f"{recuo}-> classe {no.classe} "
            f"(n={no.n}, {no.contagem})"
        )
        return

    print(
        f"{recuo}[{no.atributo}] "
        f"n={no.n}, {no.contagem}"
    )

    for v, filho in no.filhos.items():
        print(f"{recuo}  {no.atributo} = {v}:")

        imprimir(
            filho,
            recuo + "    "
        )


# ---------------------------------------------------------------------------
# 4. Leitura do CSV
# ---------------------------------------------------------------------------


def ler_csv(caminho, alvo, ignorar):
    with open(
        caminho,
        newline="",
        encoding="utf-8-sig"
    ) as f:

        linhas = list(csv.DictReader(f))

    if not linhas:
        raise ValueError("O arquivo CSV está vazio.")

    atributos = [
        c for c in linhas[0]
        if c != alvo and c not in ignorar
    ]

    # Conversão dos valores numéricos para inteiros
    for linha in linhas:
        for coluna in atributos + [alvo]:
            linha[coluna] = int(float(linha[coluna]))

    return linhas, atributos


# ---------------------------------------------------------------------------
# 5. Execução
# ---------------------------------------------------------------------------

if __name__ == "__main__":

    CAMINHO_CSV = "dados_etapa1.csv"
    ALVO = "infra_digital_basica"
    IGNORAR = ["CO_ENTIDADE"]

    # Leitura dos dados
    dados, attrs = ler_csv(
        CAMINHO_CSV,
        ALVO,
        IGNORAR
    )

    print(f"Total de exemplos: {len(dados)}")
    print(f"Total de atributos: {len(attrs)}")

    # Treinamento da árvore
    hist = []

    arvore = treinar(
        dados,
        attrs,
        ALVO,
        max_depth=None,
        historico=hist
    )

    # Entropia e ganhos da raiz
    print(f"\nH(raiz) = {hist[0]['H']:.4f}")

    print("\nGanhos de informação na raiz:")

    for a, g in sorted(
        hist[0]["ganhos"].items(),
        key=lambda kv: -kv[1]
    ):
        print(f"GI({a}) = {g:.4f}")

    # Impressão da árvore
    print("\nÁrvore de decisão:")
    imprimir(arvore)

    # Avaliação no treinamento
    print(
        f"\nAcurácia no treino: "
        f"{acuracia(arvore, dados, ALVO):.4f}"
    )






Total de exemplos: 5562
Total de atributos: 5

H(raiz) = 0.6824

Ganhos de informação na raiz:
GI(IN_LABORATORIO_INFORMATICA) = 0.1915
GI(IN_BIBLIOTECA) = 0.0989
GI(TP_LOCALIZACAO) = 0.0953
GI(TP_DEPENDENCIA) = 0.0438
GI(IN_ENERGIA_GERADOR_FOSSIL) = 0.0088

Árvore de decisão:
[IN_LABORATORIO_INFORMATICA] n=5562, {0: 4555, 1: 1007}
  IN_LABORATORIO_INFORMATICA = 0:
    [TP_LOCALIZACAO] n=4607, {0: 4238, 1: 369}
      TP_LOCALIZACAO = 1:
        [TP_DEPENDENCIA] n=1107, {0: 926, 1: 181}
          TP_DEPENDENCIA = 1:
            [IN_ENERGIA_GERADOR_FOSSIL] n=2, {1: 1, 0: 1}
              IN_ENERGIA_GERADOR_FOSSIL = 0:
                -> classe 1 (n=1, {1: 1})
              IN_ENERGIA_GERADOR_FOSSIL = 1:
                -> classe 0 (n=1, {0: 1})
          TP_DEPENDENCIA = 2:
            [IN_BIBLIOTECA] n=220, {0: 179, 1: 41}
              IN_BIBLIOTECA = 0:
                [IN_ENERGIA_GERADOR_FOSSIL] n=73, {0: 64, 1: 9}
                  IN_ENERGIA_GERADOR_FOSSIL = 0:
                    -

In [4]:

%pip install scikit-learn


Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.2 MB ? eta -:--:--
   ----- ---------------------------------- 1.0/8.2 MB 3.7 MB/s eta 0:00:02
   -------- ------------------------------- 1.8/8.2 MB 4.4 MB/s eta 0:00:02
   -------- ------------------------------- 1.8/8.2 MB 4.4 MB/s eta 0:00:02
   ---------------- ----------------------- 3.4/8.2 MB 4.0 MB/s eta 0:00:02
   ---------------- ----------------------- 3.4/8.2 MB 4.0 MB/s eta 0:00:02
   ------------------------ --------------- 5.0/8.2 MB 3.9 MB/s eta 0:00:01
   ---------------------------- ----------- 5.8/8.2 MB 4.0 MB/s eta 0:00:01
   --------------------------------- ------ 6.8/8.2 MB 4.0 MB/s eta 0:00:01
   ------------------------------------ --- 7.6/8.2 MB 4.0 MB/s eta 0:00:01
   ---------------------------------------  8.1/8.2 MB 3.9 MB/s eta 0:00:01
   ---------------------


[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: C:\Users\dougl\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [1]:
import sys
import numpy
import scipy
import sklearn

print("Python:", sys.executable)
print("NumPy:", numpy.__version__)
print("SciPy:", scipy.__version__)
print("Scikit-learn:", sklearn.__version__)

Python: C:\Users\dougl\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe
NumPy: 2.5.4
SciPy: 1.18.1
Scikit-learn: 1.9.1


In [2]:

import importlib
import inspect

import learn_decision_tree as ldt
import comparar_sklearn as cs

print("Módulos carregados com sucesso!")

print("\nFunções disponíveis:")

print("treinar:", inspect.signature(ldt.treinar))
print("prever:", inspect.signature(ldt.prever))
print("um_hot:", inspect.signature(cs.um_hot))
print("folhas:", inspect.signature(cs.folhas))

# Carregar os dados
alvo = "infra_digital_basica"

dados, attrs = ldt.ler_csv(
    "dados_etapa1.csv",
    alvo,
    ["CO_ENTIDADE"]
)

print("\nBase carregada!")
print("Total de exemplos:", len(dados))
print("Total de atributos:", len(attrs))
print("Atributos:", attrs)


Módulos carregados com sucesso!

Funções disponíveis:
treinar: (exemplos, atributos, alvo, max_depth=None, historico=None)
prever: (no, exemplo)
um_hot: (linhas, atributos, categorias=None)
folhas: (no)

Base carregada!
Total de exemplos: 5562
Total de atributos: 5
Atributos: ['IN_LABORATORIO_INFORMATICA', 'IN_BIBLIOTECA', 'TP_LOCALIZACAO', 'TP_DEPENDENCIA', 'IN_ENERGIA_GERADOR_FOSSIL']


In [4]:

# ============================================================
# 1. IMPORTAÇÕES
# ============================================================

import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.tree import DecisionTreeClassifier

import learn_decision_tree as ldt
from comparar_sklearn import folhas, um_hot


# ============================================================
# 2. CONFIGURAÇÕES
# ============================================================

k = 5
seed = 42

alvo = "infra_digital_basica"


# ============================================================
# 3. LEITURA DOS DADOS
# ============================================================

dados, attrs = ldt.ler_csv(
    "dados_etapa1.csv",
    alvo,
    ["CO_ENTIDADE"]
)

print("========== DADOS CARREGADOS ==========")
print("Total de exemplos:", len(dados))
print("Total de atributos:", len(attrs))
print("Atributos:", attrs)


# ============================================================
# 4. PREPARAÇÃO DOS DADOS
# ============================================================

y = np.array([int(d[alvo]) for d in dados])

X, _ = um_hot(dados, attrs)

print("\n========== PREPARAÇÃO ==========")
print("Formato de X:", X.shape)
print("Formato de y:", y.shape)

print("Quantidade da classe 0:", np.sum(y == 0))
print("Quantidade da classe 1:", np.sum(y == 1))


# ============================================================
# 5. VALIDAÇÃO CRUZADA
# ============================================================

skf = StratifiedKFold(
    n_splits=k,
    shuffle=True,
    random_state=seed
)

for md in (3, None):

    print("\n" + "=" * 65)

    if md is None:
        print("VALIDAÇÃO CRUZADA - PROFUNDIDADE ILIMITADA")
    else:
        print(f"VALIDAÇÃO CRUZADA - PROFUNDIDADE MÁXIMA {md}")

    print("=" * 65)

    # Resultados de cada modelo
    res = {
        "ID3 próprio": [],
        "sklearn (entropy)": [],
        "sklearn (gini)": [],
        "baseline (classe majoritária)": []
    }

    # Listas de complexidade
    folhas_p = []
    folhas_s = []

    # --------------------------------------------------------
    # 6. EXECUÇÃO DOS FOLDS
    # --------------------------------------------------------

    for i, (tr, te) in enumerate(skf.split(X, y), 1):

        # Separação dos dados de treinamento e teste
        d_tr = [dados[j] for j in tr]
        d_te = [dados[j] for j in te]

        # ----------------------------------------------------
        # ID3 PRÓPRIO
        # ----------------------------------------------------

        arv = ldt.treinar(
            d_tr,
            attrs,
            alvo,
            md
        )

        previsoes_id3 = [
            int(ldt.prever(arv, exemplo))
            for exemplo in d_te
        ]

        acc_p = np.mean(
            np.array(previsoes_id3) == y[te]
        )

        folhas_p.append(folhas(arv))

        res["ID3 próprio"].append(acc_p)

        # ----------------------------------------------------
        # SKLEARN: ENTROPIA E GINI
        # ----------------------------------------------------

        for crit in ("entropy", "gini"):

            clf = DecisionTreeClassifier(
                criterion=crit,
                max_depth=md,
                random_state=0
            )

            clf.fit(X[tr], y[tr])

            acc = clf.score(X[te], y[te])

            res[f"sklearn ({crit})"].append(acc)

            if crit == "entropy":
                folhas_s.append(clf.get_n_leaves())

        # ----------------------------------------------------
        # BASELINE: CLASSE MAJORITÁRIA
        # ----------------------------------------------------

        maj = np.bincount(y[tr]).argmax()

        acc_b = np.mean(y[te] == maj)

        res["baseline (classe majoritária)"].append(acc_b)

        # ----------------------------------------------------
        # RESULTADO DO FOLD
        # ----------------------------------------------------

        print(f"\nFold {i}")

        for nome, valores in res.items():
            print(f"{nome}: {valores[-1]:.4f}")

    # ========================================================
    # 7. RESULTADOS FINAIS
    # ========================================================

    print("\n" + "-" * 65)
    print("RESULTADOS MÉDIOS")
    print("-" * 65)

    for nome, valores in res.items():

        media = np.mean(valores)
        desvio = np.std(valores, ddof=1)

        print(
            f"{nome}: {media:.4f} ± {desvio:.4f}"
        )

    # ========================================================
    # 8. COMPLEXIDADE DAS ÁRVORES
    # ========================================================

    print("\n========== COMPLEXIDADE ==========")

    print(
        "ID3 próprio - média de folhas:",
        np.mean(folhas_p)
    )

    print(
        "Scikit-learn (entropy) - média de folhas:",
        np.mean(folhas_s)
    )


print("\n========== EXPERIMENTO CONCLUÍDO ==========")



========== DADOS CARREGADOS ==========
Total de exemplos: 5562
Total de atributos: 5
Atributos: ['IN_LABORATORIO_INFORMATICA', 'IN_BIBLIOTECA', 'TP_LOCALIZACAO', 'TP_DEPENDENCIA', 'IN_ENERGIA_GERADOR_FOSSIL']

========== PREPARAÇÃO ==========
Formato de X: (5562, 12)
Formato de y: (5562,)
Quantidade da classe 0: 4555
Quantidade da classe 1: 1007

VALIDAÇÃO CRUZADA - PROFUNDIDADE MÁXIMA 3

Fold 1
ID3 próprio: 0.8733
sklearn (entropy): 0.8733
sklearn (gini): 0.8733
baseline (classe majoritária): 0.8185

Fold 2
ID3 próprio: 0.8805
sklearn (entropy): 0.8814
sklearn (gini): 0.8814
baseline (classe majoritária): 0.8185

Fold 3
ID3 próprio: 0.8768
sklearn (entropy): 0.8768
sklearn (gini): 0.8750
baseline (classe majoritária): 0.8192

Fold 4
ID3 próprio: 0.8732
sklearn (entropy): 0.8732
sklearn (gini): 0.8732
baseline (classe majoritária): 0.8192

Fold 5
ID3 próprio: 0.8786
sklearn (entropy): 0.8786
sklearn (gini): 0.8786
baseline (classe majoritária): 0.8192

---------------------------------